# Homework 5 – Option 2: Build and Consume a Local API
**Kim Shellenberger**

This notebook builds a local REST API using Flask and then demonstrates consuming it as a client.

**Topic:** A simple book library — supports listing, looking up, and adding books.

---

## Part 1 — Server

The Flask server is launched in a background thread so the notebook stays interactive while the API runs.

In [1]:
# Install Flask and requests if not already present
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "flask", "requests", "--quiet"])
print("Flask and requests ready.")

Flask and requests ready.


In [2]:
import threading
from flask import Flask, jsonify, request

app = Flask(__name__)

# ── In-memory data store ─────────────────────────────────────────────────────
books = {
    1: {"id": 1, "title": "Book 1", "author": "Author 1", "year": 2001},
    2: {"id": 2, "title": "Book 2", "author": "Author 2", "year": 2002},
    3: {"id": 3, "title": "Book 3", "author": "Author 3", "year": 2003},
}
next_id = 4


# ── GET /books ── return all books (200) ─────────────────────────────────────
@app.route("/books", methods=["GET"])
def get_books():
    return jsonify(list(books.values())), 200


# ── GET /books/<id> ── return one book (200) or not found (404) ──────────────
@app.route("/books/<int:book_id>", methods=["GET"])
def get_book(book_id):
    book = books.get(book_id)
    if book is None:
        return jsonify({"error": f"Book with id {book_id} not found."}), 404
    return jsonify(book), 200


# ── POST /books ── add a new book (201) or reject bad input (400) ─────────────
@app.route("/books", methods=["POST"])
def add_book():
    global next_id
    data = request.get_json(silent=True)

    if not data:
        return jsonify({"error": "Request body must be JSON."}), 400

    missing = [f for f in ("title", "author", "year") if f not in data]
    if missing:
        return jsonify({"error": f"Missing required fields: {missing}"}), 400

    book = {
        "id":     next_id,
        "title":  data["title"],
        "author": data["author"],
        "year":   int(data["year"]),
    }
    books[next_id] = book
    next_id += 1
    return jsonify(book), 201


# ── Start server in a daemon thread ──────────────────────────────────────────
server_thread = threading.Thread(
    target=lambda: app.run(port=5050, debug=False, use_reloader=False),
    daemon=True,
)
server_thread.start()
print("Server running on http://127.0.0.1:5050")

Server running on http://127.0.0.1:5050


---

## Part 2 — Client

Using the requests library to call each endpoint and demonstrate every status code.

In [3]:
import time, requests

BASE = "http://127.0.0.1:5050"
time.sleep(1)  # wait for server to start
print("Client ready.")

 * Debug mode: off


 * Running on http://127.0.0.1:5050
Press CTRL+C to quit


Client ready.


In [4]:
# ── GET /books  →  200 OK ────────────────────────────────────────────────────
response = requests.get(f"{BASE}/books")
print(f"GET /books  →  {response.status_code}")
for book in response.json():
    print(f"  [{book['id']}] {book['title']} by {book['author']} ({book['year']})")

127.0.0.1 - - [19/Jun/2026 17:35:03] "GET /books HTTP/1.1" 200 -


GET /books  →  200
  [1] Book 1 by Author 1 (2001)
  [2] Book 2 by Author 2 (2002)
  [3] Book 3 by Author 3 (2003)


In [5]:
# ── GET /books/2  →  200 OK ──────────────────────────────────────────────────
response = requests.get(f"{BASE}/books/2")
print(f"GET /books/2  →  {response.status_code}")
print(" ", response.json())

127.0.0.1 - - [19/Jun/2026 17:35:03] "GET /books/2 HTTP/1.1" 200 -


GET /books/2  →  200
  {'author': 'Author 2', 'id': 2, 'title': 'Book 2', 'year': 2002}


In [6]:
# ── GET /books/99  →  404 Not Found ─────────────────────────────────────────
response = requests.get(f"{BASE}/books/99")
print(f"GET /books/99  →  {response.status_code}")
print(" ", response.json())

127.0.0.1 - - [19/Jun/2026 17:35:03] "GET /books/99 HTTP/1.1" 404 -


GET /books/99  →  404
  {'error': 'Book with id 99 not found.'}


In [7]:
# ── POST /books  →  201 Created ──────────────────────────────────────────────
new_book = {"title": "Book 4", "author": "Author 4", "year": 2004}
response = requests.post(f"{BASE}/books", json=new_book)
print(f"POST /books  →  {response.status_code}")
print(" ", response.json())

127.0.0.1 - - [19/Jun/2026 17:35:03] "POST /books HTTP/1.1" 201 -


POST /books  →  201
  {'author': 'Author 4', 'id': 4, 'title': 'Book 4', 'year': 2004}


In [8]:
# ── POST /books  →  400 Bad Request (missing fields) ─────────────────────────
bad_payload = {"title": "Incomplete Book"}  # missing author and year
response = requests.post(f"{BASE}/books", json=bad_payload)
print(f"POST /books (bad data)  →  {response.status_code}")
print(" ", response.json())

127.0.0.1 - - [19/Jun/2026 17:35:03] "POST /books HTTP/1.1" 400 -


POST /books (bad data)  →  400
  {'error': "Missing required fields: ['author', 'year']"}


In [9]:
# ── GET /books  →  verify new book was saved ─────────────────────────────────
response = requests.get(f"{BASE}/books")
print(f"GET /books  →  {response.status_code}")
print(f"Total books now: {len(response.json())}")
for book in response.json():
    print(f"  [{book['id']}] {book['title']} by {book['author']} ({book['year']})")

127.0.0.1 - - [19/Jun/2026 17:35:03] "GET /books HTTP/1.1" 200 -


GET /books  →  200
Total books now: 4
  [1] Book 1 by Author 1 (2001)
  [2] Book 2 by Author 2 (2002)
  [3] Book 3 by Author 3 (2003)
  [4] Book 4 by Author 4 (2004)


---

## Summary

| Endpoint | Method | Status Code | Meaning |
|----------|--------|-------------|----------|
| /books | GET | **200** | All books returned successfully |
| /books/<id> | GET | **200** | Single book found |
| /books/<id> | GET | **404** | Book id does not exist |
| /books | POST | **201** | New book created successfully |
| /books | POST | **400** | Request rejected — missing required fields |

Data is stored in a Python dictionary (books) in memory. The Flask server runs in a background thread so Jupyter cells can still execute while the API is live.

---

## Written Component

### 1. What my API does and why it's RESTful

I built a small book-library API in Flask with three routes: GET /books lists every book, GET /books/<id> looks up one, and POST /books adds a new one. Data lives in a Python dictionary, so it's in memory and resets when the server restarts.

I'd call this RESTful because it follows the constraints Fielding laid out when he coined the term. Each URL is a noun — /books, not /getBooks — and the HTTP method decides what happens to it, not the URL itself. The server also never remembers a client between calls. As Fielding (2000) put it, REST requires that "each request from client to server must contain all of the information necessary to understand the request, and cannot take advantage of any stored context on the server." That's exactly how my routes behave — every request stands on its own. I'm also using real HTTP status codes (`200`, `201`, `400`, `404`) instead of inventing my own error format, so the API speaks a language any HTTP client already understands.

### 2. Adding authentication and rate limiting

Right now anyone can hit any endpoint — no login, no key, nothing. OWASP (2023a) notes that "the authentication mechanism is an easy target for attackers since it's exposed to everyone," which describes my API exactly as it stands. The fix would be a decorator that checks an Authorization header before the route logic runs, rejecting anything without a valid key or token with a `401`. For something real, I'd issue short-lived JWTs at login rather than share one static key with every client.

Rate limiting would slot in the same way — a decorator (or a library like Flask-Limiter) tracking requests per client over a time window and returning `429` once they go over. The reason it matters: unlimited requests can "lead to DoS due to resource starvation" (OWASP, 2023b) — one client hammering /books could starve everyone else.

### 3. Scalability considerations

The biggest limit right now is that books lives in one process's memory — restart the server and the data's gone, and I can't run two copies of this app side by side because they wouldn't share state. Moving that dictionary to a real database (Postgres, Redis, whatever fits) fixes both problems.

The dev server itself isn't built to scale either — app.run() is single-threaded and meant for local testing, not production traffic. Swapping it for Gunicorn or uWSGI with multiple workers would be the next step. Statelessness already does a lot of the scaling work for me, though: Fielding (2000) notes that REST's constraints exist to emphasize "scalability of component interactions" — because nothing is stored between requests, any server instance can answer any request, so I can scale horizontally without worrying about sticky sessions. I'd also cache GET /books and paginate it once the library gets big, so one request doesn't have to return the whole dataset.

---

### References

Fielding, R. T. (2000). *Architectural styles and the design of network-based software architectures* (Doctoral dissertation, University of California, Irvine). https://roy.gbiv.com/pubs/dissertation/rest_arch_style.htm

OWASP Foundation. (2023a). *API2:2023 Broken authentication*. OWASP API Security Top 10. https://owasp.org/API-Security/editions/2023/en/0xa2-broken-authentication/

OWASP Foundation. (2023b). *API4:2023 Unrestricted resource consumption*. OWASP API Security Top 10. https://owasp.org/API-Security/editions/2023/en/0xa4-unrestricted-resource-consumption/